# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset using the `mlcroissant` library. We will cover metadata inspection, records loading, and basic data analysis following the Croissant schema and referencing all dataset entities by their `@id` fields.

### Dataset Source
This dataset is described by a Croissant schema hosted at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier (DOI): {getattr(metadata, 'identifier', 'N/A')}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values for subsequent data processing. This step helps understand the data structure and select which record sets and fields to analyze.

In [ ]:
# List all record sets by @id and name (if available).
print("Available Record Sets (@id and name):")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {getattr(rs, '@id', 'N/A')}, name: {getattr(rs, 'name', 'N/A')}")

# For demonstration, print fields for each record set using @id references
for rs in record_sets:
    print(f"\nFields for Record Set @id: {getattr(rs, '@id', 'N/A')} ({getattr(rs, 'name', 'N/A')}):")
    for field in rs.fields:
        print(f"  - field @id: {getattr(field, '@id', 'N/A')}, name: {getattr(field, 'name', 'N/A')}, dataType: {getattr(field, 'dataType', 'N/A')}")

## 3. Data Extraction
Extract data from the record sets identified above. Each DataFrame index is a record set `@id` and columns are field `@id`s. Select a record set and its fields using their `@id`s.

In [ ]:
# Get list of record set @id values
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]

# Create dictionary for storing DataFrames
dataframes = {}
for record_set in record_set_ids:
    # Using records generator from mlcroissant
    records = list(dataset.records(record_set=record_set))
    if records:
        dataframes[record_set] = pd.DataFrame(records)

print("Loaded DataFrames for Record Sets:")
for rid, df in dataframes.items():
    print(f"- {rid}: {df.shape[0]} records, {df.shape[1]} fields.")

# Display the columns (field @id's) for the first record set with data
if dataframes:
    primary_record_set_id = next(iter(dataframes))
    print(f"\nField (@id) columns for record set {primary_record_set_id}:")
    print(dataframes[primary_record_set_id].columns.tolist())
    display(dataframes[primary_record_set_id].head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing and cleaning steps, referring to field columns by their `@id`. Example steps include filtering records, normalizing numeric fields, and grouping. Modify the `numeric_field_id` and `group_field_id` variables to use the appropriate field `@id` values for your analysis.

In [ ]:
# Choose the record set to analyze (by @id, from previous code cell)
record_set_id = primary_record_set_id  # Update if you want a different record set
df = dataframes[record_set_id]

# Inspect and select a numeric field for analysis (field @id)
print("Available field @ids for numeric analysis:")
print(df.columns.tolist())
# Please update this value according to your data
numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.select_dtypes(include='number').empty else df.columns[0]

threshold = 10

# Filter records with numeric field above threshold
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the selected field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field, e.g., a categorical field (choose field @id)
    group_field_id = None
    # Find a likely categorical field
    for col in df.columns:
        if df[col].nunique() < 10 and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields referenced by their `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# If grouping field exists, boxplot grouped by category
if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and process the FAIR² dataset using the `mlcroissant` library, referencing record sets and fields by their `@id`. Further analysis can be performed by exploring additional field relationships and domain-specific questions guided by the Croissant schema.